# 🚀 Demistyfikacja RAG: Architektura, Systemy i Ewaluacja
**Prezentacja dla studentów AI/ML**

W poprzednich częściach omówiliśmy fundamenty wyszukiwania informacji (Information Retrieval): Bag of Words, TF-IDF, algorytm BM25 oraz gęste wektory (Dense Embeddings) i indeksowanie HNSW.
Teraz złożymy te elementy w produkcyjny system **Retrieval-Augmented Generation (RAG)** i – co najważniejsze dla inżynierów ML – nauczymy się, **jak matematycznie mierzyć jego skuteczność**.

---

## 1. Czym jest RAG i w jakim celu się go stosuje?

Wielkie Modele Językowe (LLM) posiadają ogromną wiedzę ogólną, ale mają trzy fundamentalne ograniczenia:
1. **Brak dostępu do prywatnych danych:** Nie znają wewnętrznej dokumentacji firmy, kodu źródłowego ani prywatnych baz danych.
   1. Tylko 5% informacji jest indeksowanych i dostępnych w "publicznie dostępnym" internecie.
2. **Deaktualizacja wiedzy (Knowledge Cut-off):** Ich wiedza kończy się w momencie zakończenia treningu.
    | Model Name            | Knowledge Cut-off date | Public Release Date |
    |-----------------------|------------------------|---------------------|
    | GPT-5.4               | August 31, 2025        | March 05, 2026      |
    | Claude 4.6 Sonnet     | August 2025            | January 17, 2026    |
    | Claude 4.6 Opus       | May 2025               | February 5, 2026    |
    | Gemini 3.1 Flash Lite | January 2025           | March 03, 2026      |
    | Gemini 3.1 Pro        | January 2025           | February 19, 2026   |

    źródło: https://www.allmo.ai/articles/list-of-large-language-model-cut-off-dates
3. **Halucynacje:** Gdy model czegoś nie wie, ma tendencję do generowania bardzo przekonujących, ale nieprawdziwych informacji.

**Retrieval-Augmented Generation (RAG)** to architektura, która rozwiązuje te problemy poprzez połączenie systemu wyszukiwania informacji (Retriever) z modelem generatywnym (Generator). Zamiast polegać wyłącznie na "pamięci" wag modelu, RAG dostarcza modelowi odpowiedni kontekst z zewnętrznej bazy wiedzy w czasie rzeczywistym.
        ![Architektura RAG](media/rag_architektura.png)

## 2. Jak działa system RAG?

Architektura RAG dzieli się na dwie główne fazy:

### Faza Offline: Indeksowanie (Ingestion)
1. **Parsowanie i Chunkowanie:** Dokumenty (PDF, HTML, Markdown) są dzielone na mniejsze fragmenty tekstu (tzw. chunki). Naiwne cięcie co $N$ znaków często psuje kontekst, dlatego stosuje się podziały semantyczne lub strukturalne.

    ![Podział dokumentów](media/rag_podzial_dokumentow.png)
    źródło: https://www.chitika.com/importance-text-splitting-rag/
    więcej informacji: [Strategie podziału dokumentów](/documents/strategie_podzialu_dokumentow.md)
2. **Wektoryzacja (Embedding):** Każdy chunk jest przepuszczany przez model (np. `text-embedding-3-small`), który zamienia go w gęsty wektor (np. o wymiarze 1536).
    więcej informacji: [Wektoryzacja dokumentów](documents/strategie_wektoryzacji_dokumentow.md)
3. **Przechowywanie:** Wektory oraz oryginalne teksty i metadane trafiają do wektorowej bazy danych (np. ChromaDB, Qdrant, Milvus), która zazwyczaj używa algorytmów takich jak HNSW do szybkiego wyszukiwania.

### Faza Online: Wnioskowanie (Retrieval & Generation)
1. **Zapytanie Użytkownika:** System przyjmuje pytanie użytkownika i zamienia je na wektor za pomocą tego samego modelu wektoryzującego.
2. **Wyszukiwanie (Retrieval):** Baza wektorowa wykonuje wyszukiwanie (np. używając miary podobieństwa cosinusowego) i zwraca $K$ najbardziej podobnych chunków (tzw. *Approximate Nearest Neighbors*).
3. **Augmentacja i Generowanie:** Pytanie użytkownika oraz odzyskane chunki (jako kontekst) są wstrzykiwane do promptu dla LLM-a. Model generuje odpowiedź opartą **wyłącznie** na dostarczonym kontekście.
    więcej informacji: [Podejście do generowania informacji](documents/strategie_generowania_odpowiedzi.md)

In [9]:
from dotenv import load_dotenv
!pip install -r requirements.txt


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import os
import numpy as np
import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI
from langsmith import wrappers, traceable # Added for tracking
from typing import List, Dict

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Missing OPENAI_API_KEY in .env")
client = wrappers.wrap_openai(OpenAI()) # Wrap client for LangSmith tracing

print("Biblioteki zostały załadowane poprawnie!")

Biblioteki zostały załadowane poprawnie!


## 3. Implementacja Fazy Offline: Tworzenie Bazy Wiedzy

W tym przykładzie wykorzystamy **ChromaDB** jako naszą bazę wektorową oraz model **OpenAI `text-embedding-3-small`** do generowania wektorów. Zbudujemy prostą bazę wiedzy z pojęciami ML.

In [11]:
# Zaktualizowana, bogatsza baza wiedzy dla systemu RAG
documents = [
    # --- Fundamenty i Architektura ---
    {"id": "doc1", "text": "Transformery opierają się całkowicie na mechanizmie self-attention, rezygnując z rekurencji i splotów."},
    {"id": "doc4", "text": "Retrieval-Augmented Generation (RAG) opiera modele języków na zewnętrznych bazach wiedzy w celu redukcji halucynacji i dostarczenia aktualnych informacji."},
    {"id": "doc10", "text": "Zjawisko halucynacji w LLM polega na generowaniu gramatycznie poprawnych, niezwykle przekonujących, lecz faktograficznie błędnych informacji, gdy model nie posiada odpowiedniej wiedzy w swoich wagach."},

    # --- Wyszukiwanie i Wektoryzacja (Information Retrieval) ---
    {"id": "doc2", "text": "BM25 to algorytm wyszukiwania oparty na rzadkich wektorach (bag-of-words i TF-IDF), który ocenia zbiór dokumentów na podstawie dokładnych dopasowań słów kluczowych zapytania."},
    {"id": "doc3", "text": "HNSW (Hierarchical Navigable Small World) to algorytm grafowy do przybliżonego wyszukiwania najbliższych sąsiadów (ANN). Jest standardem w bazach wektorowych ze względu na logarytmiczny czas wyszukiwania."},
    {"id": "doc6", "text": "Wektoryzacja (Embeddings) to proces konwersji tekstu na gęste wektory liczbowe o stałej długości (np. 1536 wymiarów), co pozwala komputerom mierzyć semantyczne podobieństwo między zdaniami."},
    {"id": "doc14", "text": "Bi-enkodery oddzielnie wektoryzują zapytanie i dokument. Pozwala to na szybkie odpytywanie baz wektorowych za pomocą podobieństwa kosinusowego (Cosine Similarity), w przeciwieństwie do wolniejszych Cross-enkoderów."},
    {"id": "doc5", "text": "Cross-enkodery przetwarzają zapytanie i dokument jednocześnie przez jedną sieć Transformer. Zapewniają bardzo wysoką dokładność relacyjną, ale są zbyt wolne do przeszukiwania całych baz, dlatego używa się ich w fazie Re-rankingu."},
    {"id": "doc7", "text": "Wyszukiwanie hybrydowe (Hybrid Search) łączy precyzję słów kluczowych (np. BM25) z semantycznym zrozumieniem gęstych wektorów, często wykorzystując algorytm Reciprocal Rank Fusion (RRF) do połączenia obu list wyników."},
    {"id": "doc13", "text": "Rozszerzanie zapytania (Query Expansion) to technika pre-processingu, w której LLM generuje synonimy lub alternatywne sformułowania pytania użytkownika przed wysłaniem go do bazy wektorowej."},

    # --- Chunking (Dzielenie dokumentów) ---
    {"id": "doc8", "text": "RecursiveCharacterTextSplitter to popularny algorytm z biblioteki LangChain, który dzieli tekst na chunki używając hierarchii separatorów, starając się zachować strukturę akapitów i całych zdań."},
    {"id": "doc12", "text": "Podział semantyczny (Semantic Chunking) wektoryzuje kolejne zdania dokumentu i stawia granice chunków w miejscach, gdzie drastycznie spada podobieństwo kosinusowe, co oznacza naturalną zmianę tematu przez autora."},

    # --- Ewaluacja (Measuring IR) ---
    {"id": "doc9", "text": "Mean Reciprocal Rank (MRR) to klasyczna metryka Information Retrieval, w której wynik (od 0 do 1) zależy od tego, jak wysoko na zwróconej liście znajduje się pierwszy trafny dokument."},
    {"id": "doc11", "text": "Paradygmat LLM-as-a-Judge (wykorzystywany np. we frameworku Ragas) używa silnych modeli językowych jako sędziów do oceny wygenerowanej odpowiedzi pod kątem wierności kontekstowi (Faithfulness) oraz trafności (Answer Relevance)."}
]

# 2. Inicjalizacja ChromaDB z zapisem na dysk (Persistent Storage)
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# 3. Konfiguracja funkcji embeddującej OpenAI
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.environ.get("OPENAI_API_KEY"),
    model_name="text-embedding-3-small"
)

# 4. Utworzenie kolekcji (odpowiednik tabeli w SQL)
collection = chroma_client.get_or_create_collection(
    name="ml_knowledge_base",
    embedding_function=openai_ef
)

# 5. Indeksowanie danych (Ingestion)
collection.upsert(
    ids=[doc["id"] for doc in documents],
    documents=[doc["text"] for doc in documents]
)

print(f"Pomyślnie zaindeksowano {collection.count()} dokumentów w ChromaDB.")

Pomyślnie zaindeksowano 14 dokumentów w ChromaDB.


## 4. Implementacja Fazy Online: End-to-End Pipeline

Mając zaindeksowaną wiedzę, możemy stworzyć funkcję, która:
1. Przeszuka bazę wektorową.
2. Zbuduje odpowiedni `system_prompt`.
3. Poprosi LLM-a o wygenerowanie odpowiedzi.

*Wskazówka zaawansowana:* W systemach produkcyjnych często przed krokiem 2 dodaje się **Re-ranker** (np. wspomniane wcześniej Cross-enkodery), który ponownie sortuje wyniki zwrócone przez bazę wektorową dla maksymalizacji trafności.

In [12]:
@traceable(name="RAG Pipeline")
def ask_rag_system(query: str, top_k: int = 2) -> str:
    # KROK 1: WYSZUKIWANIE (Retrieval)
    results = collection.query(
        query_texts=[query],
        n_results=top_k
    )

    retrieved_docs = results['documents'][0]
    context = "\n---\n".join(retrieved_docs)

    print(f"🔍 Odzyskano {len(retrieved_docs)} fragmenty z bazy wiedzy.")

    # KROK 2: AUGMENTACJA (Konstrukcja promptu)
    system_prompt = f"""
    Jesteś asystentem AI ds. uczenia maszynowego.
    Odpowiedz na pytanie użytkownika używając WYŁĄCZNIE poniższego kontekstu.
    Jeśli odpowiedzi nie ma w kontekście, powiedz "Nie wiem na podstawie dostarczonego kontekstu."

    KONTEKST:
    {context}
    """

    # KROK 3: GENEROWANIE (Generation)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ],
        temperature=0.0 # Ustawiamy 0, aby odpowiedź była deterministyczna
    )

    return response.choices[0].message.content

In [13]:
@traceable(name="Pure LLM (No RAG)")
def ask_pure_llm(query: str) -> str:
    """Odpytuje model językowy bazując wyłącznie na jego wyuczonej wiedzy (wagach)."""

    print("🧠 Odpytywanie czystego modelu LLM (bez bazy wektorowej)...")

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Jesteś ekspertem ds. uczenia maszynowego. Odpowiedz na pytanie krótko i zwięźle."},
            {"role": "user", "content": query}
        ],
        temperature=0.0
    )

    return response.choices[0].message.content

# --- DEMONSTRACJA PORÓWNAWCZA ---
# Wybierzmy bardzo specyficzne pytanie, na które odpowiedź znajduje się w naszych dokumentach
test_query = "W jaki sposób Semantic Chunking decyduje, gdzie postawić granicę podziału tekstu?"

print("==================================================")
print("Pytanie:", test_query)
print("==================================================")

# 1. Odpowiedź z samego LLM-a
pure_answer = ask_pure_llm(test_query)
print(f"\n❌ Odpowiedź samego LLM-a (Pure LLM):\n{pure_answer}\n")

print("--------------------------------------------------")

# 2. Odpowiedź z naszego systemu RAG (używa funkcji, którą masz w komórce 7)
rag_answer = ask_rag_system(test_query, top_k=2)
print(f"\n✅ Odpowiedź systemu RAG:\n{rag_answer}")

Pytanie: W jaki sposób Semantic Chunking decyduje, gdzie postawić granicę podziału tekstu?
🧠 Odpytywanie czystego modelu LLM (bez bazy wektorowej)...

❌ Odpowiedź samego LLM-a (Pure LLM):
Semantic Chunking decyduje o granicach podziału tekstu na podstawie analizy semantycznej, identyfikując naturalne jednostki znaczeniowe, takie jak frazy, zdania czy akapity. Wykorzystuje techniki takie jak analiza składniowa, rozpoznawanie encji oraz modele językowe, aby zrozumieć kontekst i relacje między słowami, co pozwala na efektywne wyodrębnienie sensownych fragmentów tekstu.

--------------------------------------------------
🔍 Odzyskano 2 fragmenty z bazy wiedzy.

✅ Odpowiedź systemu RAG:
Semantic Chunking decyduje, gdzie postawić granicę podziału tekstu, analizując podobieństwo kosinusowe między kolejnymi zdaniami dokumentu i stawiając granice chunków w miejscach, gdzie drastycznie spada to podobieństwo, co wskazuje na naturalną zmianę tematu przez autora.


In [14]:
# Testujemy system!
query = "Jakie są główne różnice między bi-enkoderami a cross-enkoderami i dlaczego używamy ich razem?"
answer = ask_rag_system(query, top_k=3)
print("\n🤖 Odpowiedź Systemu:\n", answer)

🔍 Odzyskano 3 fragmenty z bazy wiedzy.

🤖 Odpowiedź Systemu:
 Bi-enkodery oddzielnie wektoryzują zapytanie i dokument, co pozwala na szybkie odpytywanie baz wektorowych za pomocą podobieństwa kosinusowego. Z kolei cross-enkodery przetwarzają zapytanie i dokument jednocześnie przez jedną sieć Transformer, co zapewnia bardzo wysoką dokładność relacyjną, ale są zbyt wolne do przeszukiwania całych baz. 

Używamy ich razem, ponieważ bi-enkodery umożliwiają szybkie przeszukiwanie, a cross-enkodery zapewniają dokładniejsze wyniki w fazie re-rankingowej.


## 5. Mierzenie Information Retrieval (IR) w systemach RAG

Skąd wiemy, czy nasz system RAG działa dobrze? Jako inżynierowie nie zgadujemy – my to mierzymy.
Ocena RAG dzieli się na ocenę modułu **Retrieval** (czy znajdujemy dobre dokumenty?) oraz modułu **Generation** (czy model poprawnie używa dokumentów?).

Aby ocenić Retriever, potrzebujemy zbioru testowego z przypisanym *Ground Truth*, czyli par `(zapytanie, ID_oczekiwanego_dokumentu)`.

### Klasyczne Metryki IR:

1. **Recall@K (Czułość):** Czy odpowiedni dokument znalazł się gdziekolwiek w top $K$ wynikach wyszukiwania? Jeśli dokumentu nie ma w kontekście, LLM nie ma szans na poprawne odpowiedzenie na pytanie. Jest to metryka binarna (1 - jest, 0 - brak).

2. **MRR (Mean Reciprocal Rank):**
Ta metryka ocenia, *jak wysoko* na liście znajduje się pierwszy trafny dokument.
$$MRR = \frac{1}{|Q|} \sum_{i=1}^{|Q|} \frac{1}{rank_i}$$
Gdzie $rank_i$ to pozycja trafnego dokumentu w wynikach.
* Jeśli dokument jest na 1 miejscu, wynik to $1.0$.
* Jeśli na 2 miejscu, wynik to $0.5$.
* Jeśli na 3 miejscu, wynik to $0.33$.

3. **NDCG (Normalized Discounted Cumulative Gain):**
Bardziej zaawansowana metryka stosowana w systemach, gdzie dokumenty mają różne wagi/stopnie trafności (np. oceny od 0 do 3), a nie tylko binarne pasuje/nie pasuje.

In [15]:
# Zbiór testowy Ground Truth: [(zapytanie, oczekiwane_id_dokumentu)]
eval_dataset = [
    ("Jaki algorytm pozwala na szybkie wyszukiwanie wektorów?", "doc3"), # HNSW
    ("Na jakiej zasadzie działa funkcja BM25?", "doc2"), # BM25
    ("Która architektura rezygnuje ze splotów na rzecz self-attention?", "doc1"), # Transformery
    ("Jaki model przetwarza zapytanie i dokument w tym samym czasie?", "doc5"), # Cross-encoders
    ("Czym jest zjawisko halucynacji w kontekście AI?", "doc10"), # Halucynacje
    ("Jakie podejście łączy precyzję słów kluczowych z wektorami?", "doc7"), # Hybrid Search
    ("Na czym polega wektoryzowanie całych zdań w celu uniknięcia złego podziału?", "doc12") # Semantic chunking
]

def evaluate_retriever(dataset, k=3):
    mrr_scores = []
    recall_scores = []

    for query, expected_id in dataset:
        # Pobieramy top K dokumentów dla zapytania
        results = collection.query(
            query_texts=[query],
            n_results=k
        )
        retrieved_ids = results['ids'][0]

        # Obliczanie Recall@K
        hit = 1 if expected_id in retrieved_ids else 0
        recall_scores.append(hit)

        # Obliczanie Reciprocal Rank
        rr = 0
        if hit:
            rank = retrieved_ids.index(expected_id) + 1
            rr = 1.0 / rank
        mrr_scores.append(rr)

    return {
        f"Recall@{k}": np.mean(recall_scores),
        "MRR": np.mean(mrr_scores)
    }

# Uruchomienie ewaluacji
metrics = evaluate_retriever(eval_dataset, k=3)
print("📊 Wyniki Ewaluacji Retrievera:")
for metric, value in metrics.items():
    print(f"{metric}: {value:.2f}")

📊 Wyniki Ewaluacji Retrievera:
Recall@3: 1.00
MRR: 0.93


## 6. Zaawansowana Ewaluacja: Paradygmat "LLM-as-a-Judge"

Tradycyjne metryki IR oceniają jakość wyszukiwania, ale jak ocenić otwarty tekst generowany przez LLM? Klasyczne metryki NLP, takie jak BLEU czy ROUGE, nie sprawdzają się w ocenie sensu semantycznego.

więcej informacji: [Mierzenie IR](/documents/mierzenie_ir.md)

Obecnie rynkowym standardem (stosowanym we frameworkach takich jak **Ragas** czy **TruLens**) jest wykorzystanie potężnego modelu (np. GPT-4) w roli sędziego, który ocenia tzw. **Triadę RAG**:

1. **Context Relevance (Trafność kontekstu):** Czy wyszukany kontekst zawiera przydatne informacje?
2. **Faithfulness (Wierność / Brak halucynacji):** Czy wygenerowana odpowiedź opiera się *wyłącznie* na dostarczonym kontekście?
3. **Answer Relevance (Trafność odpowiedzi):** Czy odpowiedź faktycznie rozwiązuje problem użytkownika zadany w zapytaniu?

Poniżej implementujemy prostego sędziego oceniającego Wierność (Faithfulness).

In [16]:
@traceable(name="LLM-as-a-Judge")
def check_faithfulness(question: str, context: str, generated_answer: str) -> Dict:
    """Używa LLM do oceny, czy wygenerowana odpowiedź halucynuje poza podany kontekst."""

    judge_prompt = f"""
    Mając podany KONTEKST oraz ODPOWIEDŹ wygenerowaną przez system AI,
    określ czy ODPOWIEDŹ jest ściśle oparta na KONTEKŚCIE.
    Jeśli odpowiedź zawiera fakty, których NIE MA w kontekście, wypisz dokładnie "HALUCYNACJA".
    Jeśli odpowiedź jest w pełni poparta kontekstem, wypisz dokładnie "WIERNA".

    KONTEKST: {context}

    ODPOWIEDŹ: {generated_answer}
    """

    # Jako sędziego używamy zazwyczaj potężniejszego i droższego modelu (np. GPT-4o)
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": judge_prompt}],
        temperature=0.0
    )

    score_text = response.choices[0].message.content.upper()
    score = "WIERNA" if "WIERNA" in score_text and "HALUCYNACJA" not in score_text else "HALUCYNACJA"
    
    return {
        "score": score,
        "answer": generated_answer
    }

# Przetestujmy to w praktyce!
mock_context = "RAG opiera LLM-y na zewnętrznych bazach wiedzy."

faithful_ans = "Architektura RAG korzysta z zewnętrznych baz do dostarczania wiedzy modelom."
hallucinated_ans = "Architektura RAG korzysta z zewnętrznych baz do dostarczania wiedzy modelom. RAG został wymyślony w firmie Meta w 2020 roku."

print("Test 1 (Odpowiedź Poprawna):", check_faithfulness("Co to RAG?", mock_context, faithful_ans)['score'])
print("Test 2 (Odpowiedź z Halucynacją):", check_faithfulness("Co to RAG?", mock_context, hallucinated_ans)['score'])

Test 1 (Odpowiedź Poprawna): WIERNA
Test 2 (Odpowiedź z Halucynacją): HALUCYNACJA


### 🎉 Podsumowanie
Podczas tej sesji przeszliśmy od teorii TF-IDF i wektorów do działającego systemu RAG. W zastosowaniach produkcyjnych (nad którymi pracujemy jako inżynierowie AI) ten schemat rozbudowuje się o:
* **Hybrid Search:** Łączenie wyszukiwania rzadkiego (BM25 - idealne do nazw własnych i ID) z wektorowym (Semantic Search - idealne do pojęć).
* **Cross-Encoders:** Używanie cięższych modeli tylko do re-rankingu top-10 wyników.
* **Query Transformation:** Automatyczne przepisywanie źle sformułowanych pytań użytkownika przed uderzeniem do bazy.

Pamiętajcie: Architektura RAG jest w dużej mierze zadaniem inżynieryjnym polegającym na zarządzaniu jakością danych oraz precyzją ich wyszukiwania (IR), a nie tylko podpinaniem API z modelem językowym!

## Materiały dodatkowe
[Modern Information Retrieval Evaluation In The RAG Era](https://www.youtube.com/watch?v=Trps2swgeOg)
- zapytania są dłuższe niż w google, bardziej szczegółowe.
- Użytkownik jest cierpliwy, czeka dłużej na odpowiedź, oczekuje wszystkiego w jednym miejscu, nie klika na najpopularniejszy wynik z google.
- Mało informacji dostępnych publicznie (95% treści nie jest indeksowanych w google - intranet, portale uniwersyteckie itp.)

[IBM: What is Retrieval-Augmented Generation (RAG)?](https://www.youtube.com/watch?v=T-D1OfcDW1M)

[IBM: Is RAG Still Needed? Choosing the Best Approach for LLMs](https://www.youtube.com/watch?v=UabBYexBD4k)
LLM Context Window jest ogromny. Możemy w nim zmieścić trylogia władcy pierścieni i zostanie jeszcze miejsce na hobbita.


[Cutoff dates](https://www.allmo.ai/articles/list-of-large-language-model-cut-off-dates)

[Every RAG Strategy Explained in 13 Minutes (No Fluff)](https://www.youtube.com/watch?v=tLMViADvSNE)
[Every RAG Strategy - Code Examples](https://github.com/coleam00/ottomator-agents/tree/main/all-rag-strategies)

[Learn RAG From Scratch – Python AI Tutorial from a LangChain Engineer
](https://www.youtube.com/watch?v=sVcwVQRHIc8)